# RQ2 Final Frozen-Test Ablation Study

Final contract-verified AFP ablation evaluation for AdaPruner-KGQA.

This notebook contains the final frozen-test feature-family and selector-component ablation implementation used in the study.


In [ ]:

# ==================================================================================================
# FINAL ONE-CELL FROZEN-TEST ABLATION STUDY — FRESH-SESSION SAFE / CONTRACT-VERIFIED
# ==================================================================================================
# For a future fresh session, the safe order is: 17G-R0 → 17G → 17I/final recovery → revised Cell 18 → this cell.
# Run this ONLY after the frozen Cell-17I runtime has been recovered successfully.
# It does NOT depend on the old ablation setup/A1/A2/A3 Python variables.
#
# FINAL TEST VARIANTS
#   1. Full AFP
#   2. w/o Semantic        : remove Feature-v2 indices 0..7   -> 19-D retrained Cell-15A scorer
#   3. w/o Path Context    : remove Feature-v2 indices 8..14  -> 20-D retrained Cell-15A scorer
#   4. w/o Structural      : remove Feature-v2 indices 15..22 -> 19-D retrained Cell-15A scorer
#   5. w/o Progress        : remove Feature-v2 indices 23..26 -> 23-D retrained Cell-15A scorer
#   6. w/o Uncertainty     : Cell-15B exact definition; gamma=gamma_min on non-ties
#   7. w/o Temperature     : Cell-15B exact definition; T=1.0, uncertainty adaptation retained
#   8. Fixed Threshold     : frozen Cell-17 control
#
# SCIENTIFIC RULES
#   - NO TEST training
#   - NO TEST tuning
#   - NO TEST model selection
#   - feature ablations use ONLY the already-retrained Cell-15A checkpoints
#   - selector ablations use ONLY the predeclared Cell-15B definitions
#   - Full AFP / Fixed Threshold must exactly reproduce the frozen Cell-17 TEST controls
#   - final-hop protection and singleton bypass remain in the frozen traversal
#
# CRITICAL SOFTWARE FIXES
#   - frozen get_group_logits calls UPPERCASE AFP_RUNTIME_SCORE_GROUP
#   - this cell patches that exact binding, never the obsolete lowercase alias
#   - routed feature scorers return the same dict contract as the frozen scorer
#   - hard routing/dimension gates prove the 19/20/19/23-D scorers were actually executed
# ==================================================================================================

from pathlib import Path
from collections import defaultdict
import copy
import gzip
import hashlib
import inspect
import json
import math
import re
import time

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F


print("=" * 150)
print("FINAL ONE-CELL FROZEN-TEST ABLATION — FRESH-SESSION SAFE / CONTRACT-VERIFIED")
print("=" * 150)


# ==================================================================================================
# 0. IMMUTABLE SCIENTIFIC / RUNTIME GATES
# ==================================================================================================

EXPECTED_FREEZE_SHA = (
    "bf7ef1783aa96fe483dfcd9819393a7ec"
    "1c2bb9936a842d99e2a4ad085113116"
)


assert globals().get(
    "CELL17_COMPLETE",
    False
) is True, (
    "STOP: CELL17_COMPLETE is not True. "
    "Recover the final Cell-17I runtime first."
)


assert globals().get(
    "FINAL_RQ2_TEST_COMPARISON_COMPLETE",
    False
) is True, (
    "STOP: FINAL_RQ2_TEST_COMPARISON_COMPLETE is not True."
)


if "FINAL_AFP_FREEZE_SHA256" in globals():

    assert (
        FINAL_AFP_FREEZE_SHA256
        ==
        EXPECTED_FREEZE_SHA
    ), (
        "STOP: frozen AFP scientific configuration hash changed."
    )


for _name in [
    "run_controlled_dataset",
    "traverse_controlled_method",
]:

    assert (
        _name in globals()
        and
        callable(
            globals()[
                _name
            ]
        )
    ), (
        f"STOP: missing runtime function {_name}"
    )


FROZEN_RUN = (
    run_controlled_dataset
)


FROZEN_TRAVERSE = (
    traverse_controlled_method
)


RUN_GLOBALS = (
    FROZEN_RUN.__globals__
)


TRAVERSE_GLOBALS = (
    FROZEN_TRAVERSE.__globals__
)


assert (
    "get_group_logits"
    in TRAVERSE_GLOBALS
)


assert callable(
    TRAVERSE_GLOBALS[
        "get_group_logits"
    ]
)


GET_GROUP_LOGITS = (
    TRAVERSE_GLOBALS[
        "get_group_logits"
    ]
)


GET_LOGITS_GLOBALS = (
    GET_GROUP_LOGITS.__globals__
)


# ------------------------------------------------------------------------------------------
# CRITICAL CONTRACT:
# get_group_logits must call uppercase AFP_RUNTIME_SCORE_GROUP.
# ------------------------------------------------------------------------------------------

assert (
    "AFP_RUNTIME_SCORE_GROUP"
    in GET_GROUP_LOGITS.__code__.co_names
), (
    "STOP: get_group_logits does not reference "
    "uppercase AFP_RUNTIME_SCORE_GROUP."
)


assert (
    "AFP_RUNTIME_SCORE_GROUP"
    in GET_LOGITS_GLOBALS
)


assert (
    "controlled_selection"
    in TRAVERSE_GLOBALS
)


assert (
    "traverse_controlled_method"
    in RUN_GLOBALS
)


FROZEN_SCORE_GROUP = (
    GET_LOGITS_GLOBALS[
        "AFP_RUNTIME_SCORE_GROUP"
    ]
)


FROZEN_SELECTION = (
    TRAVERSE_GLOBALS[
        "controlled_selection"
    ]
)


FROZEN_RUN_TRAVERSE = (
    RUN_GLOBALS[
        "traverse_controlled_method"
    ]
)


assert callable(
    FROZEN_SCORE_GROUP
)


assert callable(
    FROZEN_SELECTION
)


assert (
    FROZEN_RUN_TRAVERSE
    is
    FROZEN_TRAVERSE
)


assert (
    getattr(
        FROZEN_SCORE_GROUP,
        "__name__",
        ""
    )
    ==
    "afp_runtime_score_group"
), (
    "STOP: uppercase AFP_RUNTIME_SCORE_GROUP is not "
    "the frozen Cell-17 scorer. Restore Cell-17I "
    "before running this cell."
)


assert (
    getattr(
        FROZEN_SELECTION,
        "__name__",
        ""
    )
    ==
    "controlled_selection"
)


print(
    "Frozen Cell-17I runtime: PASSED"
)

print(
    "get_group_logits -> AFP_RUNTIME_SCORE_GROUP: PASSED"
)

print(
    "TEST training/tuning/model-selection: FORBIDDEN"
)


# ==================================================================================================
# 1. RESOLVE THE EXACT FROZEN TEST OBJECTS
#    NO OLD ABLATION VARIABLES REQUIRED
# ==================================================================================================

def resolve_global(
    candidates,
    description
):

    for name in candidates:

        if name in globals():

            return (
                globals()[
                    name
                ],
                name
            )

    raise RuntimeError(
        f"STOP: could not resolve {description}.\n"
        f"Tried: {candidates}"
    )


WEB_PLAN_ROWS, WEB_PLAN_NAME = (
    resolve_global(
        [
            "webqsp_test_plan_rows",
            "WEBQSP_TEST_PLAN_ROWS_17",
            "WEB_TEST_PLAN_ROWS_17",
        ],
        "WebQSP TEST planning rows"
    )
)


CWQ_PLAN_ROWS, CWQ_PLAN_NAME = (
    resolve_global(
        [
            "cwq_test_plan_rows",
            "CWQ_TEST_PLAN_ROWS_17",
            "CWQ_TEST_PLAN_ROWS_17G",
        ],
        "CWQ TEST planning rows"
    )
)


WEB_Q_ROWS, WEB_Q_NAME = (
    resolve_global(
        [
            "WEBQSP_TEST_QUESTION_ROWS_17",
            "WEBQSP_TEST_QUESTION_ROWS_17G",
        ],
        "WebQSP TEST question rows"
    )
)


CWQ_Q_ROWS, CWQ_Q_NAME = (
    resolve_global(
        [
            "CWQ_TEST_QUESTION_ROWS_17",
            "CWQ_TEST_QUESTION_ROWS_17G",
        ],
        "CWQ TEST question rows"
    )
)


WEB_ROG_REF, WEB_ROG_NAME = (
    resolve_global(
        [
            "WEB_TEST_ROG_17G",
            "WEB_TEST_ROG_17",
        ],
        "WebQSP frozen RoG TEST reference"
    )
)


CWQ_ROG_REF, CWQ_ROG_NAME = (
    resolve_global(
        [
            "CWQ_TEST_ROG_17G",
            "CWQ_TEST_ROG_17",
        ],
        "CWQ frozen RoG TEST reference"
    )
)


WEB_METHODS, WEB_METHODS_NAME = (
    resolve_global(
        [
            "WEB_METHODS_17G",
            "WEB_METHODS_17",
        ],
        "WebQSP frozen method specs"
    )
)


CWQ_METHODS, CWQ_METHODS_NAME = (
    resolve_global(
        [
            "CWQ_METHODS_17G",
            "CWQ_METHODS_17",
        ],
        "CWQ frozen method specs"
    )
)


assert (
    len(
        WEB_PLAN_ROWS
    )
    ==
    len(
        WEB_Q_ROWS
    )
    ==
    1628
)


assert (
    len(
        CWQ_PLAN_ROWS
    )
    ==
    len(
        CWQ_Q_ROWS
    )
    ==
    3531
)


# ------------------------------------------------------------------------------------------
# Exact frozen Cell-17 TEST references.
# These act as deterministic Full-AFP / Fixed-Threshold controls.
# ------------------------------------------------------------------------------------------

TEST_REFERENCE = {

    "webqsp": {

        "n_questions":
            1628,

        "rog_edges":
            2320856,

        "rog_reachable":
            1369,

        "full_afp_edges":
            2268901,

        "full_afp_reachable":
            1358,

        "fixed_threshold_edges":
            2292980,

        "fixed_threshold_reachable":
            1341,
    },

    "cwq": {

        "n_questions":
            3531,

        "rog_edges":
            5819125,

        "rog_reachable":
            2422,

        "full_afp_edges":
            5525531,

        "full_afp_reachable":
            2391,

        "fixed_threshold_edges":
            5794478,

        "fixed_threshold_reachable":
            2422,
    },
}


assert (
    int(
        WEB_ROG_REF[
            "edges_examined"
        ]
    )
    ==
    TEST_REFERENCE[
        "webqsp"
    ][
        "rog_edges"
    ]
)


assert (
    int(
        WEB_ROG_REF[
            "reachable_questions"
        ]
    )
    ==
    TEST_REFERENCE[
        "webqsp"
    ][
        "rog_reachable"
    ]
)


assert (
    int(
        CWQ_ROG_REF[
            "edges_examined"
        ]
    )
    ==
    TEST_REFERENCE[
        "cwq"
    ][
        "rog_edges"
    ]
)


assert (
    int(
        CWQ_ROG_REF[
            "reachable_questions"
        ]
    )
    ==
    TEST_REFERENCE[
        "cwq"
    ][
        "rog_reachable"
    ]
)


print(
    "\nFrozen TEST objects: PASSED"
)

print(
    " WebQSP:",
    WEB_PLAN_NAME,
    "|",
    WEB_Q_NAME,
    "|",
    WEB_ROG_NAME,
    "|",
    WEB_METHODS_NAME
)

print(
    " CWQ:   ",
    CWQ_PLAN_NAME,
    "|",
    CWQ_Q_NAME,
    "|",
    CWQ_ROG_NAME,
    "|",
    CWQ_METHODS_NAME
)


# ==================================================================================================
# 2. RECOVER FROZEN AFP / FIXED-THRESHOLD METHOD SPECS
# ==================================================================================================

def find_family_spec(
    obj,
    family
):

    if isinstance(
        obj,
        dict
    ):

        if (
            obj.get(
                "family"
            )
            ==
            family
        ):

            return copy.deepcopy(
                obj
            )

        for v in obj.values():

            found = find_family_spec(
                v,
                family
            )

            if found is not None:

                return found


    elif isinstance(
        obj,
        (
            list,
            tuple,
        )
    ):

        for v in obj:

            found = find_family_spec(
                v,
                family
            )

            if found is not None:

                return found


    return None


WEB_AFP_SPEC = (
    find_family_spec(
        WEB_METHODS,
        "afp"
    )
)


CWQ_AFP_SPEC = (
    find_family_spec(
        CWQ_METHODS,
        "afp"
    )
)


WEB_THRESHOLD_SPEC = (
    find_family_spec(
        WEB_METHODS,
        "fixed_threshold"
    )
)


CWQ_THRESHOLD_SPEC = (
    find_family_spec(
        CWQ_METHODS,
        "fixed_threshold"
    )
)


assert (
    WEB_AFP_SPEC
    is not None
)


assert (
    CWQ_AFP_SPEC
    is not None
)


assert (
    WEB_THRESHOLD_SPEC
    is not None
)


assert (
    CWQ_THRESHOLD_SPEC
    is not None
)


assert (
    abs(
        float(
            WEB_AFP_SPEC[
                "T"
            ]
        )
        -
        0.05
    )
    <=
    1e-12
)


assert (
    abs(
        float(
            CWQ_AFP_SPEC[
                "T"
            ]
        )
        -
        0.10
    )
    <=
    1e-12
)


assert (
    abs(
        float(
            WEB_AFP_SPEC[
                "gamma_min"
            ]
        )
        -
        0.10
    )
    <=
    1e-12
)


assert (
    abs(
        float(
            CWQ_AFP_SPEC[
                "gamma_min"
            ]
        )
        -
        0.10
    )
    <=
    1e-12
)


print(
    "Frozen selector values: PASSED"
)

print(
    " WebQSP T=0.05 gamma_min=0.10"
)

print(
    " CWQ    T=0.10 gamma_min=0.10"
)


# ==================================================================================================
# 3. LOCATE + VERIFY CELL-15A / CELL-15B FROZEN ARTIFACTS
# ==================================================================================================

BASE_WORK = Path(
    "/kaggle/working/step3_rq2_dev_v1"
)


MIGRATION_BASE = Path(
    "/kaggle/input/datasets/"
    "sabitahnaf/"
    "adapruner-kgqa-migration/"
    "step3_rq2_dev_v1"
)


def first_existing(
    paths,
    description
):

    for p in paths:

        if p.exists():

            return p

    raise FileNotFoundError(
        description
        +
        " not found:\n"
        +
        "\n".join(
            map(
                str,
                paths
            )
        )
    )


def sha256_file(
    path
):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        for block in iter(
            lambda:
                f.read(
                    1024
                    *
                    1024
                ),
            b""
        ):

            h.update(
                block
            )

    return h.hexdigest()


FEATURE_ABL_ROOT = (
    first_existing(
        [
            BASE_WORK
            /
            "13_feature_ablations",

            MIGRATION_BASE
            /
            "13_feature_ablations",
        ],
        "Cell-15A feature-ablation root"
    )
)


SELECTOR_ABL_ROOT = (
    first_existing(
        [
            BASE_WORK
            /
            "14_selector_ablations_and_final_freeze",

            MIGRATION_BASE
            /
            "14_selector_ablations_and_final_freeze",
        ],
        "Cell-15B selector/freeze root"
    )
)


FREEZE_JSON_PATH = (
    SELECTOR_ABL_ROOT
    /
    "final_afp_development_freeze.json"
)


FREEZE_SHA_PATH = (
    SELECTOR_ABL_ROOT
    /
    "final_afp_development_freeze.sha256"
)


assert (
    FREEZE_JSON_PATH.exists()
)


assert (
    FREEZE_SHA_PATH.exists()
)


with open(
    FREEZE_JSON_PATH,
    "r",
    encoding="utf-8"
) as f:

    FREEZE_JSON = json.load(
        f
    )


# ------------------------------------------------------------------------------------------
# Recompute the exact canonical Cell-15B scientific freeze SHA.
# ------------------------------------------------------------------------------------------

canonical_freeze = json.dumps(
    FREEZE_JSON,
    sort_keys=True,
    separators=(
        ",",
        ":"
    ),
    ensure_ascii=False
)


actual_freeze_sha = hashlib.sha256(
    canonical_freeze.encode(
        "utf-8"
    )
).hexdigest()


recorded_freeze_sha = (
    FREEZE_SHA_PATH
    .read_text(
        encoding="utf-8"
    )
    .strip()
)


assert (
    actual_freeze_sha
    ==
    EXPECTED_FREEZE_SHA
)


assert (
    recorded_freeze_sha
    ==
    EXPECTED_FREEZE_SHA
)


assert (
    FREEZE_JSON[
        "features"
    ][
        "dimension"
    ]
    ==
    27
)


assert (
    FREEZE_JSON[
        "features"
    ][
        "final_decision"
    ]
    ==
    "retain_full_feature_v2"
)


assert (
    FREEZE_JSON[
        "selector"
    ][
        "webqsp"
    ]
    ==
    {
        "T":
            0.05,

        "gamma_min":
            0.1,
    }
)


assert (
    FREEZE_JSON[
        "selector"
    ][
        "cwq"
    ]
    ==
    {
        "T":
            0.1,

        "gamma_min":
            0.1,
    }
)


assert (
    FREEZE_JSON[
        "selector"
    ][
        "uncertainty"
    ]
    ==
    "normalized_entropy"
)


assert (
    FREEZE_JSON[
        "selector"
    ][
        "fully_tied_behavior"
    ]
    ==
    "retain_all"
)


assert (
    FREEZE_JSON[
        "selector"
    ][
        "cutoff_tie_behavior"
    ]
    ==
    "expand_cutoff_ties"
)


assert (
    FREEZE_JSON[
        "selector"
    ][
        "further_hyperparameter_search_allowed"
    ]
    is False
)


assert (
    FREEZE_JSON[
        "leakage_controls"
    ][
        "selector_tuning"
    ]
    ==
    "validation_only"
)


assert (
    FREEZE_JSON[
        "leakage_controls"
    ][
        "test_used_for_training"
    ]
    is False
)


assert (
    FREEZE_JSON[
        "leakage_controls"
    ][
        "test_used_for_model_selection"
    ]
    is False
)


assert (
    FREEZE_JSON[
        "leakage_controls"
    ][
        "test_used_for_hyperparameter_tuning"
    ]
    is False
)


# ------------------------------------------------------------------------------------------
# Prove that the two selector/component ablations were predeclared in Cell-15B.
# ------------------------------------------------------------------------------------------

for dataset in [
    "webqsp",
    "cwq",
]:

    selector_csv = (
        SELECTOR_ABL_ROOT
        /
        f"{dataset}_selector_component_ablation.csv"
    )


    assert (
        selector_csv.exists()
    )


    sdf = pd.read_csv(
        selector_csv
    )


    assert (
        "variant"
        in sdf.columns
    )


    observed = set(
        sdf[
            "variant"
        ].astype(
            str
        )
    )


    assert {
        "full_afp",
        "no_uncertainty_adaptation",
        "no_temperature_scaling",
    }.issubset(
        observed
    )


print(
    "\nCell-15B canonical freeze hash: PASSED"
)

print(
    " ",
    EXPECTED_FREEZE_SHA
)

print(
    "Cell-15A root:",
    FEATURE_ABL_ROOT
)

print(
    "Cell-15B root:",
    SELECTOR_ABL_ROOT
)


# ==================================================================================================
# 4. EXACT 27-D FEATURE SCHEMA + FEATURE REMOVAL CONTRACT
# ==================================================================================================

FEATURE_NAMES_27 = [

    # ----------------------------------------------------------------------------------------------
    # Semantic family [0:8] = 8 dimensions
    # ----------------------------------------------------------------------------------------------

    "sem_q_candidate",
    "sem_candidate_surface_available",
    "sem_q_current_entity",
    "sem_current_surface_available",
    "sem_q_current_relation",
    "sem_q_full_plan",
    "sem_q_remaining_suffix",
    "sem_candidate_current_relation",

    # ----------------------------------------------------------------------------------------------
    # Path-context family [8:15] = 7 dimensions
    # ----------------------------------------------------------------------------------------------

    "path_q_prefix_entity_mean",
    "path_candidate_prefix_entity_mean",
    "path_prefix_surface_fraction",
    "path_candidate_repeats_entity",
    "path_candidate_occurrence_fraction",
    "path_unique_entity_ratio",
    "path_relation_repeat_fraction_before",

    # ----------------------------------------------------------------------------------------------
    # Structural family [15:23] = 8 dimensions
    # ----------------------------------------------------------------------------------------------

    "struct_log_candidate_count",
    "struct_log_unique_candidate_entities",
    "struct_log_contributing_parents",
    "struct_log_parent_fanout",
    "struct_parent_frontier_share",
    "struct_log_endpoint_multiplicity",
    "struct_endpoint_frontier_share",
    "struct_duplicate_endpoint_ratio",

    # ----------------------------------------------------------------------------------------------
    # Progress family [23:27] = 4 dimensions
    # ----------------------------------------------------------------------------------------------

    "prog_hop_fraction",
    "prog_remaining_fraction",
    "prog_log_plan_length",
    "prog_penultimate_indicator",
]


assert (
    len(
        FEATURE_NAMES_27
    )
    ==
    27
)


FEATURE_VARIANTS = {

    # ==============================================================================================
    # Remove semantic [0..7].
    # Keep [8..26] = 19 dimensions.
    # ==============================================================================================

    "w/o Semantic": {

        "internal":
            "minus_semantic",

        "file":
            "minus_semantic.pt",

        "removed":
            list(
                range(
                    0,
                    8
                )
            ),

        "keep":
            list(
                range(
                    8,
                    27
                )
            ),

        "dim":
            19,
    },


    # ==============================================================================================
    # Remove path context [8..14].
    # Keep [0..7] + [15..26] = 20 dimensions.
    # ==============================================================================================

    "w/o Path Context": {

        "internal":
            "minus_path",

        "file":
            "minus_path.pt",

        "removed":
            list(
                range(
                    8,
                    15
                )
            ),

        "keep":
            (
                list(
                    range(
                        0,
                        8
                    )
                )
                +
                list(
                    range(
                        15,
                        27
                    )
                )
            ),

        "dim":
            20,
    },


    # ==============================================================================================
    # Remove structural [15..22].
    # Keep [0..14] + [23..26] = 19 dimensions.
    # ==============================================================================================

    "w/o Structural": {

        "internal":
            "minus_structural",

        "file":
            "minus_structural.pt",

        "removed":
            list(
                range(
                    15,
                    23
                )
            ),

        "keep":
            (
                list(
                    range(
                        0,
                        15
                    )
                )
                +
                list(
                    range(
                        23,
                        27
                    )
                )
            ),

        "dim":
            19,
    },


    # ==============================================================================================
    # Remove progress [23..26].
    # Keep [0..22] = 23 dimensions.
    # ==============================================================================================

    "w/o Progress": {

        "internal":
            "minus_progress",

        "file":
            "minus_progress.pt",

        "removed":
            list(
                range(
                    23,
                    27
                )
            ),

        "keep":
            list(
                range(
                    0,
                    23
                )
            ),

        "dim":
            23,
    },
}


# ------------------------------------------------------------------------------------------
# Mathematical partition gates.
# ------------------------------------------------------------------------------------------

for variant, cfg in (
    FEATURE_VARIANTS.items()
):

    assert (
        len(
            cfg[
                "keep"
            ]
        )
        ==
        cfg[
            "dim"
        ]
    )


    assert (
        len(
            set(
                cfg[
                    "keep"
                ]
            )
        )
        ==
        cfg[
            "dim"
        ]
    )


    assert not (
        set(
            cfg[
                "keep"
            ]
        )
        &
        set(
            cfg[
                "removed"
            ]
        )
    )


    assert sorted(
        set(
            cfg[
                "keep"
            ]
        )
        |
        set(
            cfg[
                "removed"
            ]
        )
    ) == list(
        range(
            27
        )
    )


# ------------------------------------------------------------------------------------------
# Cross-check dimensions against the frozen Cell-15A development evidence inside Cell-15B freeze.
# ------------------------------------------------------------------------------------------

for dataset in [
    "webqsp",
    "cwq",
]:

    frozen_rows = {

        r[
            "variant"
        ]:
            r

        for r in (
            FREEZE_JSON[
                "feature_ablation_results"
            ][
                dataset
            ]
        )
    }


    for variant, cfg in (
        FEATURE_VARIANTS.items()
    ):

        assert (
            cfg[
                "internal"
            ]
            in frozen_rows
        )


        assert (
            int(
                frozen_rows[
                    cfg[
                        "internal"
                    ]
                ][
                    "input_dim"
                ]
            )
            ==
            cfg[
                "dim"
            ]
        )


# ------------------------------------------------------------------------------------------
# Load Cell-15A manifest if available.
# It gives checkpoint SHA + feature-column provenance.
# ------------------------------------------------------------------------------------------

CELL15A_MANIFEST_PATH = (
    FEATURE_ABL_ROOT
    /
    "cell15a_feature_family_retraining_ablation_manifest.json"
)


CELL15A_MANIFEST = None


if CELL15A_MANIFEST_PATH.exists():

    with open(
        CELL15A_MANIFEST_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        CELL15A_MANIFEST = json.load(
            f
        )


    assert (
        CELL15A_MANIFEST[
            "methodology"
        ]
        ==
        "remove_feature_family_then_retrain"
    )


    assert (
        CELL15A_MANIFEST[
            "test_examples_accessed"
        ]
        is False
    )


    for variant, cfg in (
        FEATURE_VARIANTS.items()
    ):

        m = (
            CELL15A_MANIFEST[
                "variants"
            ][
                cfg[
                    "internal"
                ]
            ]
        )


        assert (
            list(
                m[
                    "feature_columns"
                ]
            )
            ==
            cfg[
                "keep"
            ]
        )


        assert (
            int(
                m[
                    "input_dim"
                ]
            )
            ==
            cfg[
                "dim"
            ]
        )


print(
    "\n"
    +
    "=" * 150
)

print(
    "FEATURE REMOVAL CONTRACT"
)

print(
    "=" * 150
)


for variant, cfg in (
    FEATURE_VARIANTS.items()
):

    print(
        f"{variant:18s}: "
        f"remove {cfg['removed']} "
        f"-> keep {cfg['dim']:2d}-D"
    )


    print(
        "  removed features:",
        [
            FEATURE_NAMES_27[
                i
            ]
            for i
            in cfg[
                "removed"
            ]
        ]
    )


# ==================================================================================================
# 5. LOAD + HARD-VERIFY THE EIGHT RETRAINED CELL-15A CHECKPOINTS
# ==================================================================================================

def to_numpy(
    x
):

    if torch.is_tensor(
        x
    ):

        return (
            x.detach()
            .cpu()
            .numpy()
        )

    return np.asarray(
        x
    )


def recover_standardizer_state(
    state,
    expected_dim
):

    assert isinstance(
        state,
        dict
    ), (
        "standardizer_state must be a dict"
    )


    mean = None
    std = None


    for k, v in (
        state.items()
    ):

        arr = to_numpy(
            v
        )


        if arr.shape != (
            expected_dim,
        ):

            continue


        kl = str(
            k
        ).lower()


        if "mean" in kl:

            mean = np.asarray(
                arr,
                dtype=np.float32
            )


        if (
            "std"
            in kl
            or
            "scale"
            in kl
        ):

            std = np.asarray(
                arr,
                dtype=np.float32
            )


    assert (
        mean is not None
        and
        std is not None
    )


    assert (
        mean.shape
        ==
        std.shape
        ==
        (
            expected_dim,
        )
    )


    assert np.all(
        np.isfinite(
            mean
        )
    )


    assert np.all(
        np.isfinite(
            std
        )
    )


    assert np.all(
        std
        >
        0.0
    ), (
        "Checkpoint standardizer contains non-positive std"
    )


    return (
        mean,
        std
    )


def recover_mlp_state(
    state_dict,
    expected_dim
):

    assert isinstance(
        state_dict,
        dict
    )


    tensors = {

        k:
            v.detach().cpu()

        for k, v
        in state_dict.items()

        if torch.is_tensor(
            v
        )
    }


    matrices = [
        (
            k,
            v
        )
        for k, v
        in tensors.items()
        if v.ndim == 2
    ]


    first = [
        (
            k,
            v
        )
        for k, v
        in matrices
        if (
            int(
                v.shape[
                    1
                ]
            )
            ==
            expected_dim
            and
            int(
                v.shape[
                    0
                ]
            )
            >
            1
        )
    ]


    assert (
        len(
            first
        )
        ==
        1
    ), (
        f"Expected one first layer for D={expected_dim}, "
        f"got {[(k, tuple(v.shape)) for k,v in first]}"
    )


    w1_key, W1 = (
        first[
            0
        ]
    )


    hidden = int(
        W1.shape[
            0
        ]
    )


    second = [
        (
            k,
            v
        )
        for k, v
        in matrices
        if tuple(
            v.shape
        )
        ==
        (
            1,
            hidden
        )
    ]


    assert (
        len(
            second
        )
        ==
        1
    ), (
        "Expected one output layer, got "
        f"{[(k, tuple(v.shape)) for k,v in second]}"
    )


    w2_key, W2 = (
        second[
            0
        ]
    )


    def find_bias(
        weight_key,
        size
    ):

        direct = (
            weight_key[
                :-6
            ]
            +
            "bias"

            if weight_key.endswith(
                "weight"
            )

            else
            None
        )


        if (
            direct
            in tensors
            and
            tensors[
                direct
            ].ndim
            ==
            1
            and
            int(
                tensors[
                    direct
                ].numel()
            )
            ==
            size
        ):

            return tensors[
                direct
            ]


        prefix = (
            weight_key.rsplit(
                ".",
                1
            )[
                0
            ]
        )


        candidates = [
            v

            for k, v
            in tensors.items()

            if (
                k.startswith(
                    prefix
                )
                and
                "bias"
                in k.lower()
                and
                v.ndim
                ==
                1
                and
                int(
                    v.numel()
                )
                ==
                size
            )
        ]


        assert (
            len(
                candidates
            )
            ==
            1
        ), (
            f"Could not uniquely resolve bias for {weight_key}"
        )


        return candidates[
            0
        ]


    return {

        "W1":
            W1.float(),

        "b1":
            find_bias(
                w1_key,
                hidden
            ).float(),

        "W2":
            W2.float(),

        "b2":
            find_bias(
                w2_key,
                1
            ).float(),

        "hidden_dim":
            hidden,

        "w1_key":
            w1_key,

        "w2_key":
            w2_key,
    }


FEATURE_BUNDLES = {
    "webqsp": {},
    "cwq": {},
}


FEATURE_CHECKPOINT_SHA = {
    "webqsp": {},
    "cwq": {},
}


for dataset in [
    "webqsp",
    "cwq",
]:

    expected_hidden = (
        32
        if dataset
        ==
        "webqsp"
        else
        64
    )


    for variant, cfg in (
        FEATURE_VARIANTS.items()
    ):

        path = (
            FEATURE_ABL_ROOT
            /
            dataset
            /
            cfg[
                "file"
            ]
        )


        assert (
            path.exists()
        ), (
            f"Missing Cell-15A checkpoint: {path}"
        )


        file_sha = sha256_file(
            path
        )


        FEATURE_CHECKPOINT_SHA[
            dataset
        ][
            variant
        ] = file_sha


        # ------------------------------------------------------------------------------------------
        # Match current checkpoint bytes to the Cell-15A manifest when available.
        # ------------------------------------------------------------------------------------------

        if CELL15A_MANIFEST is not None:

            manifest_ckpt = (
                CELL15A_MANIFEST[
                    "ablation_checkpoints"
                ][
                    dataset
                ][
                    cfg[
                        "internal"
                    ]
                ]
            )


            assert (
                manifest_ckpt[
                    "sha256"
                ]
                ==
                file_sha
            ), (
                f"Checkpoint SHA mismatch: "
                f"{dataset} {variant}"
            )


        ckpt = torch.load(
            path,
            map_location="cpu",
            weights_only=False
        )


        # ------------------------------------------------------------------------------------------
        # Frozen Cell-15A metadata contracts.
        # ------------------------------------------------------------------------------------------

        assert (
            ckpt[
                "dataset"
            ]
            ==
            dataset
        )


        assert (
            ckpt[
                "variant"
            ]
            ==
            cfg[
                "internal"
            ]
        )


        assert (
            ckpt[
                "feature_version"
            ]
            ==
            FREEZE_JSON[
                "features"
            ][
                "version"
            ]
        )


        # ==========================================================================================
        # MOST IMPORTANT CHECK:
        # checkpoint itself must declare exactly the kept columns.
        # ==========================================================================================

        assert (
            list(
                ckpt[
                    "feature_columns"
                ]
            )
            ==
            cfg[
                "keep"
            ]
        ), (
            f"Wrong kept columns: "
            f"{dataset} {variant}"
        )


        assert (
            int(
                ckpt[
                    "input_dim"
                ]
            )
            ==
            cfg[
                "dim"
            ]
        )


        assert (
            int(
                ckpt[
                    "hidden_dim"
                ]
            )
            ==
            expected_hidden
        )


        assert (
            int(
                ckpt[
                    "seed"
                ]
            )
            ==
            42
        )


        assert (
            int(
                ckpt[
                    "epochs"
                ]
            )
            ==
            80
        )


        assert (
            abs(
                float(
                    ckpt[
                        "learning_rate"
                    ]
                )
                -
                1e-3
            )
            <=
            1e-15
        )


        assert (
            abs(
                float(
                    ckpt[
                        "weight_decay"
                    ]
                )
                -
                1e-4
            )
            <=
            1e-15
        )


        assert (
            ckpt[
                "loss_name"
            ]
            ==
            "branch_bce"
        )


        assert (
            ckpt[
                "ablation_specific_tuning"
            ]
            is False
        )


        assert (
            ckpt[
                "test_examples_accessed"
            ]
            is False
        )


        mean, std = (
            recover_standardizer_state(
                ckpt[
                    "standardizer_state"
                ],
                cfg[
                    "dim"
                ]
            )
        )


        mlp = (
            recover_mlp_state(
                ckpt[
                    "model_state_dict"
                ],
                cfg[
                    "dim"
                ]
            )
        )


        # ==========================================================================================
        # The first MLP layer MUST physically have D inputs.
        # This prevents the old silent 27-D/full-scorer behavior.
        # ==========================================================================================

        assert (
            mlp[
                "hidden_dim"
            ]
            ==
            expected_hidden
        )


        assert (
            tuple(
                mlp[
                    "W1"
                ].shape
            )
            ==
            (
                expected_hidden,
                cfg[
                    "dim"
                ]
            )
        )


        assert (
            tuple(
                mlp[
                    "W2"
                ].shape
            )
            ==
            (
                1,
                expected_hidden
            )
        )


        FEATURE_BUNDLES[
            dataset
        ][
            variant
        ] = {

            "path":
                path,

            "sha256":
                file_sha,

            "input_dim":
                cfg[
                    "dim"
                ],

            "keep":
                np.asarray(
                    cfg[
                        "keep"
                    ],
                    dtype=np.int64
                ),

            "mean":
                mean,

            "std":
                std,

            **mlp,
        }


        print(
            f" {dataset:7s} | "
            f"{variant:18s} | "
            f"D={cfg['dim']:2d} | "
            f"H={expected_hidden:2d} | "
            f"SHA={file_sha[:12]}..."
        )


print(
    "All Cell-15A reduced-dimensional "
    "checkpoint contracts: PASSED"
)


# ==================================================================================================
# 6. RECOVER THE EXACT FROZEN FEATURE EXTRACTOR + SEMANTIC ENCODER
# ==================================================================================================

SCORE_GLOBALS = (
    FROZEN_SCORE_GROUP.__globals__
)


assert (
    "AFP_RUNTIME_FEATURE_EXTRACTOR"
    in SCORE_GLOBALS
)


assert (
    "AFP_RUNTIME_SEMANTIC_ENCODER"
    in SCORE_GLOBALS
)


FEATURE_EXTRACTOR = (
    SCORE_GLOBALS[
        "AFP_RUNTIME_FEATURE_EXTRACTOR"
    ]
)


SEMANTIC_ENCODER = (
    SCORE_GLOBALS[
        "AFP_RUNTIME_SEMANTIC_ENCODER"
    ]
)


assert callable(
    FEATURE_EXTRACTOR
)


print(
    "\nFrozen Feature-v2 extractor/"
    "semantic encoder: PASSED"
)


# ==================================================================================================
# 7. ROUTING / DIMENSION AUDIT STATE
# ==================================================================================================

SCORE_CALLS = defaultdict(
    int
)


SELECTOR_CALLS = defaultdict(
    int
)


OBSERVED_DIMS = defaultdict(
    set
)


LOGIT_DIFF_SEEN = defaultdict(
    bool
)


RANK_DIFF_SEEN = defaultdict(
    bool
)


CURRENT_DATASET = None
CURRENT_VARIANT = None
CURRENT_PATH_HANDLE = None
CURRENT_PATH_COUNTS = None


# ==================================================================================================
# 8. EXACT 27-D EXTRACTION
#    -> COLUMN REMOVAL
#    -> D-D STANDARDIZATION
#    -> D-INPUT RETRAINED MLP
# ==================================================================================================

def extract_full_27d(
    question_id,
    question,
    plan,
    hop,
    candidate_rows
):

    X = FEATURE_EXTRACTOR(

        question_id=
            question_id,

        question=
            question,

        plan=
            list(
                plan
            ),

        hop=
            int(
                hop
            ),

        candidate_rows=
            candidate_rows,

        semantic_encoder=
            SEMANTIC_ENCODER,

        entity_name_map=
            None,
    )


    X = np.asarray(
        X,
        dtype=np.float32
    )


    assert (
        X.shape
        ==
        (
            len(
                candidate_rows
            ),
            27
        )
    ), (
        f"Feature-v2 extractor returned {X.shape}, "
        f"expected {(len(candidate_rows), 27)}"
    )


    assert np.all(
        np.isfinite(
            X
        )
    )


    return X


def score_retrained_feature_ablation(
    dataset,
    variant,
    question_id,
    question,
    plan,
    hop,
    candidate_rows
):

    cfg = (
        FEATURE_VARIANTS[
            variant
        ]
    )


    bundle = (
        FEATURE_BUNDLES[
            dataset
        ][
            variant
        ]
    )


    # ------------------------------------------------------------------------------------------
    # Step A: exact frozen full Feature-v2 vector = [N, 27].
    # ------------------------------------------------------------------------------------------

    X27 = (
        extract_full_27d(
            question_id=
                question_id,

            question=
                question,

            plan=
                plan,

            hop=
                hop,

            candidate_rows=
                candidate_rows
        )
    )


    # ==========================================================================================
    # Step B: ACTUAL FEATURE REMOVAL.
    #
    # Semantic      -> 19-D
    # Path Context  -> 20-D
    # Structural    -> 19-D
    # Progress      -> 23-D
    #
    # This happens BEFORE the ablation standardizer and BEFORE the MLP.
    # ==========================================================================================

    Xred = np.asarray(

        X27[
            :,
            bundle[
                "keep"
            ]
        ],

        dtype=np.float32
    )


    assert (
        Xred.shape
        ==
        (
            len(
                candidate_rows
            ),
            cfg[
                "dim"
            ]
        )
    )


    OBSERVED_DIMS[
        (
            dataset,
            variant
        )
    ].add(
        int(
            Xred.shape[
                1
            ]
        )
    )


    # ------------------------------------------------------------------------------------------
    # Step C: use the TRAIN-only standardizer retrained for THIS reduced feature space.
    # ------------------------------------------------------------------------------------------

    Z = (
        (
            Xred
            -
            bundle[
                "mean"
            ]
        )
        /
        bundle[
            "std"
        ]
    ).astype(
        np.float32
    )


    assert (
        Z.shape
        ==
        Xred.shape
    )


    assert np.all(
        np.isfinite(
            Z
        )
    )


    # ------------------------------------------------------------------------------------------
    # Step D: exact tiny AFP MLP inference:
    #
    # ReLU(W1 z + b1) -> W2 h + b2
    #
    # W1 physically has 19 / 20 / 19 / 23 inputs depending on variant.
    # ------------------------------------------------------------------------------------------

    Zt = torch.as_tensor(
        Z,
        dtype=torch.float32,
        device="cpu"
    )


    with torch.inference_mode():

        h = F.relu(
            Zt
            @
            bundle[
                "W1"
            ].T
            +
            bundle[
                "b1"
            ]
        )


        logits = (
            h
            @
            bundle[
                "W2"
            ].T
            +
            bundle[
                "b2"
            ]
        ).reshape(
            -1
        )


    logits = (
        logits
        .detach()
        .cpu()
        .numpy()
        .astype(
            np.float32
        )
    )


    assert (
        logits.shape
        ==
        (
            len(
                candidate_rows
            ),
        )
    )


    assert np.all(
        np.isfinite(
            logits
        )
    )


    # ==========================================================================================
    # CRITICAL API CONTRACT:
    # get_group_logits does:
    #
    #     output = AFP_RUNTIME_SCORE_GROUP(...)
    #     logits = output["logits"]
    #
    # Therefore a bare NumPy vector is NOT valid.
    # ==========================================================================================

    return {

        "features":
            Xred,

        "standardized_features":
            Z,

        "logits":
            logits,

        "input_dim":
            int(
                cfg[
                    "dim"
                ]
            ),

        "kept_indices":
            tuple(
                int(
                    i
                )
                for i
                in cfg[
                    "keep"
                ]
            ),
    }


# ==================================================================================================
# 9. EXACT CELL-15B w/o-UNCERTAINTY SELECTOR
# ==================================================================================================

TIE_ATOL = (
    1e-8
)


def stable_softmax_15b(
    logits,
    temperature
):

    x = np.asarray(
        logits,
        dtype=np.float64
    )


    T = float(
        temperature
    )


    assert (
        T
        >
        0.0
    )


    values = (
        x
        /
        T
    )


    values = (
        values
        -
        np.max(
            values
        )
    )


    exp_values = np.exp(
        values
    )


    denominator = float(
        np.sum(
            exp_values
        )
    )


    assert (
        denominator
        >
        0.0
        and
        np.isfinite(
            denominator
        )
    )


    return (
        exp_values
        /
        denominator
    )


def normalized_entropy_15b(
    probabilities
):

    p = np.asarray(
        probabilities,
        dtype=np.float64
    )


    n = len(
        p
    )


    if n <= 1:

        return 0.0


    safe = np.clip(
        p,
        1e-12,
        1.0
    )


    entropy = -float(
        np.sum(
            safe
            *
            np.log(
                safe
            )
        )
    )


    return float(
        np.clip(
            entropy
            /
            math.log(
                n
            ),
            0.0,
            1.0
        )
    )


def all_tied_15b(
    logits
):

    x = np.asarray(
        logits,
        dtype=np.float64
    )


    if len(
        x
    ) <= 1:

        return True


    return bool(
        (
            np.max(
                x
            )
            -
            np.min(
                x
            )
        )
        <=
        TIE_ATOL
    )


def tie_expanded_top_k_15b(
    logits,
    requested_k
):

    x = np.asarray(
        logits,
        dtype=np.float64
    )


    n = len(
        x
    )


    k = int(
        requested_k
    )


    assert (
        1
        <=
        k
        <=
        n
    )


    if k >= n:

        return list(
            range(
                n
            )
        )


    order = np.argsort(
        -x,
        kind="stable"
    )


    cutoff = x[
        order[
            k - 1
        ]
    ]


    return [

        i

        for i, score
        in enumerate(
            x
        )

        if (
            score
            >
            cutoff

            or

            np.isclose(
                score,
                cutoff,
                atol=TIE_ATOL,
                rtol=0.0
            )
        )
    ]


def select_no_uncertainty_15b(
    logits,
    selected_T,
    gamma_min
):

    x = np.asarray(
        logits,
        dtype=np.float64
    )


    n = len(
        x
    )


    assert (
        n
        >
        1
    )


    # ==========================================================================================
    # Exact Cell-15B complete-tie behavior:
    # retain all, uncertainty=1, gamma=1.
    # ==========================================================================================

    if all_tied_15b(
        x
    ):

        return {

            "selected_indices":
                list(
                    range(
                        n
                    )
                ),

            "requested_budget":
                n,

            "retained_count":
                n,

            "uncertainty":
                1.0,

            "gamma":
                1.0,

            "fully_tied":
                True,

            "selection_type":
                "afp_without_uncertainty_adaptation",
        }


    # ------------------------------------------------------------------------------------------
    # Temperature remains unchanged in this ablation.
    # ------------------------------------------------------------------------------------------

    probabilities = stable_softmax_15b(
        x,
        selected_T
    )


    # ------------------------------------------------------------------------------------------
    # Cell-15B still MEASURES normalized entropy diagnostically.
    # It only removes uncertainty from the gamma adaptation.
    # ------------------------------------------------------------------------------------------

    measured_uncertainty = (
        normalized_entropy_15b(
            probabilities
        )
    )


    # ==========================================================================================
    # ACTUAL ABLATION:
    #
    # Full:
    # gamma = gamma_min + u * (1 - gamma_min)
    #
    # w/o Uncertainty:
    # gamma = gamma_min
    # ==========================================================================================

    gamma = float(
        gamma_min
    )


    order = np.argsort(
        -x,
        kind="stable"
    )


    cumulative = np.cumsum(
        probabilities[
            order
        ]
    )


    requested_budget = int(
        np.searchsorted(
            cumulative,
            gamma,
            side="left"
        )
        +
        1
    )


    requested_budget = min(
        max(
            requested_budget,
            1
        ),
        n
    )


    selected = (
        tie_expanded_top_k_15b(
            x,
            requested_budget
        )
    )


    return {

        "selected_indices":
            selected,

        "requested_budget":
            requested_budget,

        "retained_count":
            len(
                selected
            ),

        "uncertainty":
            float(
                measured_uncertainty
            ),

        "gamma":
            gamma,

        "fully_tied":
            False,

        "selection_type":
            "afp_without_uncertainty_adaptation",
    }


# ==================================================================================================
# 10. CORRECT UPPERCASE SCORER ROUTER + SELECTOR ROUTER
# ==================================================================================================

def final_ablation_score_router(
    dataset_name,
    question_id,
    question,
    plan,
    hop,
    candidate_rows
):

    dataset = str(
        dataset_name
    ).lower()


    variant = (
        CURRENT_VARIANT
    )


    assert (
        dataset
        ==
        CURRENT_DATASET
    )


    assert (
        variant
        is not None
    )


    SCORE_CALLS[
        (
            dataset,
            variant
        )
    ] += 1


    # ==============================================================================================
    # Four retrained reduced-dimensional Cell-15A feature ablations.
    # ==============================================================================================

    if variant in (
        FEATURE_VARIANTS
    ):

        out = (
            score_retrained_feature_ablation(

                dataset=
                    dataset,

                variant=
                    variant,

                question_id=
                    question_id,

                question=
                    question,

                plan=
                    plan,

                hop=
                    hop,

                candidate_rows=
                    candidate_rows
            )
        )


        # ------------------------------------------------------------------------------------------
        # SOFTWARE AUDIT ONLY:
        # prove the reduced scorer is not silently the full scorer.
        #
        # This comparison never changes the graph, policy, threshold,
        # hyperparameters, labels, or selected branches.
        # ------------------------------------------------------------------------------------------

        key = (
            dataset,
            variant
        )


        if not (
            LOGIT_DIFF_SEEN[
                key
            ]
        ):

            full = (
                FROZEN_SCORE_GROUP(

                    dataset_name=
                        dataset_name,

                    question_id=
                        question_id,

                    question=
                        question,

                    plan=
                        plan,

                    hop=
                        hop,

                    candidate_rows=
                        candidate_rows
                )
            )


            full_logits = np.asarray(
                full[
                    "logits"
                ],
                dtype=np.float32
            )


            ablated_logits = np.asarray(
                out[
                    "logits"
                ],
                dtype=np.float32
            )


            assert (
                full_logits.shape
                ==
                ablated_logits.shape
            )


            if not np.array_equal(
                full_logits,
                ablated_logits
            ):

                LOGIT_DIFF_SEEN[
                    key
                ] = True


            if not np.array_equal(

                np.argsort(
                    -full_logits,
                    kind="stable"
                ),

                np.argsort(
                    -ablated_logits,
                    kind="stable"
                )
            ):

                RANK_DIFF_SEEN[
                    key
                ] = True


        return out


    # ==============================================================================================
    # Full AFP, selector ablations, and Fixed Threshold all use the exact
    # frozen 27-D Cell-17 scorer.
    # ==============================================================================================

    return (
        FROZEN_SCORE_GROUP(

            dataset_name=
                dataset_name,

            question_id=
                question_id,

            question=
                question,

            plan=
                plan,

            hop=
                hop,

            candidate_rows=
                candidate_rows
        )
    )


def final_ablation_selector_router(
    dataset_name,
    method_spec,
    question_id,
    plan_index,
    hop,
    active_prefixes,
    logits
):

    dataset = str(
        dataset_name
    ).lower()


    variant = (
        CURRENT_VARIANT
    )


    assert (
        dataset
        ==
        CURRENT_DATASET
    )


    SELECTOR_CALLS[
        (
            dataset,
            variant
        )
    ] += 1


    # ==============================================================================================
    # Exact Cell-15B no-uncertainty adaptation.
    # ==============================================================================================

    if variant == (
        "w/o Uncertainty"
    ):

        return (
            select_no_uncertainty_15b(

                logits=
                    logits,

                selected_T=
                    float(
                        method_spec[
                            "T"
                        ]
                    ),

                gamma_min=
                    float(
                        method_spec[
                            "gamma_min"
                        ]
                    )
            )
        )


    # ==============================================================================================
    # Full AFP / feature ablations / w/o Temperature / Fixed Threshold
    # use the exact frozen selector.
    #
    # For w/o Temperature, its method spec is changed ONLY to T=1.0 below.
    # ==============================================================================================

    return (
        FROZEN_SELECTION(

            dataset_name=
                dataset_name,

            method_spec=
                method_spec,

            question_id=
                question_id,

            plan_index=
                plan_index,

            hop=
                hop,

            active_prefixes=
                active_prefixes,

            logits=
                logits
        )
    )


# ==================================================================================================
# 11. CLEAN DEDICATED OUTPUT ROOT + RETRIEVED-PATH CAPTURE
# ==================================================================================================

ABL_ROOT = (
    BASE_WORK
    /
    "15_final_frozen_test"
    /
    "rq2_ablation_test_FINAL_VERIFIED"
)


# ------------------------------------------------------------------------------------------
# Never mix an earlier/incomplete run with this verified run.
# Preserve an existing verified directory by renaming it.
# ------------------------------------------------------------------------------------------

if ABL_ROOT.exists():

    backup = (
        ABL_ROOT.with_name(
            ABL_ROOT.name
            +
            "_previous_"
            +
            time.strftime(
                "%Y%m%d_%H%M%S"
            )
        )
    )


    ABL_ROOT.rename(
        backup
    )


    print(
        "\nExisting FINAL_VERIFIED output moved to:",
        backup
    )


PATH_ROOT = (
    ABL_ROOT
    /
    "retrieved_paths"
)


QUESTION_ROOT = (
    ABL_ROOT
    /
    "per_question"
)


PATH_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


QUESTION_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


def jsonable(
    x
):

    if (
        x is None
        or
        isinstance(
            x,
            (
                str,
                int,
                float,
                bool,
            )
        )
    ):

        return x


    if isinstance(
        x,
        np.generic
    ):

        return x.item()


    if torch.is_tensor(
        x
    ):

        return (
            x.detach()
            .cpu()
            .tolist()
        )


    if isinstance(
        x,
        np.ndarray
    ):

        return x.tolist()


    if isinstance(
        x,
        dict
    ):

        return {

            str(
                k
            ):
                jsonable(
                    v
                )

            for k, v
            in x.items()
        }


    if isinstance(
        x,
        (
            list,
            tuple,
            set,
        )
    ):

        return [
            jsonable(
                v
            )
            for v
            in x
        ]


    return str(
        x
    )


def capture_traverse(
    *args,
    **kwargs
):

    global CURRENT_PATH_COUNTS


    sig = inspect.signature(
        FROZEN_TRAVERSE
    )


    bound = sig.bind_partial(
        *args,
        **kwargs
    )


    bound.apply_defaults()


    result = (
        FROZEN_TRAVERSE(
            *args,
            **kwargs
        )
    )


    assert isinstance(
        result,
        dict
    )


    assert (
        "final_prefixes"
        in result
    )


    paths = (
        result[
            "final_prefixes"
        ]
    )


    assert isinstance(
        paths,
        (
            list,
            tuple,
        )
    )


    assert (
        CURRENT_PATH_HANDLE
        is not None
    )


    qid = str(
        bound.arguments.get(
            "question_id"
        )
    )


    record = {

        "dataset":
            CURRENT_DATASET,

        "variant":
            CURRENT_VARIANT,

        "question_id":
            qid,

        "plan_index":
            jsonable(
                bound.arguments.get(
                    "plan_index"
                )
            ),

        "plan":
            jsonable(
                bound.arguments.get(
                    "plan"
                )
            ),

        "retrieved_path_count":
            int(
                len(
                    paths
                )
            ),

        "retrieved_paths":
            jsonable(
                paths
            ),
    }


    CURRENT_PATH_HANDLE.write(
        json.dumps(
            record,
            ensure_ascii=False,
            separators=(
                ",",
                ":"
            )
        )
        +
        "\n"
    )


    CURRENT_PATH_COUNTS[
        qid
    ] = (
        CURRENT_PATH_COUNTS.get(
            qid,
            0
        )
        +
        int(
            len(
                paths
            )
        )
    )


    return result


# ==================================================================================================
# 12. RUN ONE DATASET × VARIANT
# ==================================================================================================

VARIANTS = [

    "Full AFP",

    "w/o Semantic",

    "w/o Path Context",

    "w/o Structural",

    "w/o Progress",

    "w/o Uncertainty",

    "w/o Temperature",

    "Fixed Threshold",
]


ALL_SUMMARY_ROWS = []
ALL_PATH_FILES = []
ALL_PERQ_FILES = []


def safe_name(
    x
):

    return (
        re.sub(
            r"[^a-z0-9]+",
            "_",
            str(
                x
            ).lower()
        )
        .strip(
            "_"
        )
    )


def make_variant_spec(
    variant,
    afp_spec,
    threshold_spec
):

    if variant == (
        "Fixed Threshold"
    ):

        spec = copy.deepcopy(
            threshold_spec
        )

    else:

        spec = copy.deepcopy(
            afp_spec
        )


    spec[
        "method"
    ] = variant


    spec[
        "seed"
    ] = None


    # ==============================================================================================
    # Exact Cell-15B no-temperature ablation:
    #
    # T := 1.0
    #
    # Everything else is untouched.
    # The normal uncertainty-adaptive selector remains active.
    # ==============================================================================================

    if variant == (
        "w/o Temperature"
    ):

        spec[
            "T"
        ] = 1.0


    return spec


def run_one(
    dataset,
    variant,
    plan_rows,
    q_rows,
    rog_ref,
    afp_spec,
    threshold_spec
):

    global CURRENT_DATASET
    global CURRENT_VARIANT
    global CURRENT_PATH_HANDLE
    global CURRENT_PATH_COUNTS


    ref = (
        TEST_REFERENCE[
            dataset
        ]
    )


    CURRENT_DATASET = (
        dataset
    )


    CURRENT_VARIANT = (
        variant
    )


    CURRENT_PATH_COUNTS = {}


    spec = (
        make_variant_spec(
            variant=
                variant,

            afp_spec=
                afp_spec,

            threshold_spec=
                threshold_spec
        )
    )


    path_file = (
        PATH_ROOT
        /
        (
            f"{dataset}_"
            f"{safe_name(variant)}_"
            "retrieved_paths.jsonl.gz"
        )
    )


    perq_file = (
        QUESTION_ROOT
        /
        (
            f"{dataset}_"
            f"{safe_name(variant)}_"
            "per_question.csv"
        )
    )


    print(
        "\n"
        +
        "=" * 150
    )


    print(
        f"{dataset.upper()} — {variant}"
    )


    print(
        "=" * 150
    )


    start = time.time()


    # ==============================================================================================
    # Every dataset × variant gets its OWN run_controlled_dataset call.
    #
    # Therefore each run starts with a fresh per-question score_cache.
    # A full-AFP logit cache cannot leak into a reduced-dimensional scorer.
    # ==============================================================================================

    with gzip.open(
        path_file,
        "wt",
        encoding="utf-8"
    ) as fh:

        CURRENT_PATH_HANDLE = (
            fh
        )


        result = (
            FROZEN_RUN(

                dataset_name=
                    dataset,

                planning_rows=
                    plan_rows,

                question_rows=
                    q_rows,

                rog_reference=
                    rog_ref,

                method_specs=[
                    spec
                ]
            )
        )


    CURRENT_PATH_HANDLE = None


    elapsed = (
        time.time()
        -
        start
    )


    # ------------------------------------------------------------------------------------------
    # Exact frozen run_controlled_dataset contract.
    # ------------------------------------------------------------------------------------------

    assert isinstance(
        result,
        tuple
    )


    assert (
        len(
            result
        )
        ==
        2
    )


    run_df, perq_df = (
        result
    )


    assert isinstance(
        run_df,
        pd.DataFrame
    )


    assert isinstance(
        perq_df,
        pd.DataFrame
    )


    assert (
        len(
            run_df
        )
        ==
        1
    )


    assert (
        len(
            perq_df
        )
        ==
        ref[
            "n_questions"
        ]
    )


    row = (
        run_df.iloc[
            0
        ]
    )


    edges = int(
        row[
            "edges_examined"
        ]
    )


    reachable = int(
        row[
            "reachable_questions"
        ]
    )


    ssr = float(
        row[
            "ssr"
        ]
    )


    ar = float(
        row[
            "answer_retention"
        ]
    )


    # ------------------------------------------------------------------------------------------
    # Independent metric recomputation.
    # ------------------------------------------------------------------------------------------

    assert (
        abs(
            ssr
            -
            (
                1.0
                -
                edges
                /
                ref[
                    "rog_edges"
                ]
            )
        )
        <=
        1e-12
    )


    assert (
        abs(
            ar
            -
            (
                reachable
                /
                ref[
                    "rog_reachable"
                ]
            )
        )
        <=
        1e-12
    )


    # ==============================================================================================
    # FULL AFP CONTROL MUST REPRODUCE CELL-17 EXACTLY.
    # ==============================================================================================

    if variant == (
        "Full AFP"
    ):

        assert (
            edges
            ==
            ref[
                "full_afp_edges"
            ]
        ), (
            f"{dataset}: Full AFP edge control mismatch"
        )


        assert (
            reachable
            ==
            ref[
                "full_afp_reachable"
            ]
        ), (
            f"{dataset}: Full AFP reachability control mismatch"
        )


        print(
            "Frozen Cell-17 Full AFP control: EXACT"
        )


    # ==============================================================================================
    # FIXED THRESHOLD CONTROL MUST REPRODUCE CELL-17 EXACTLY.
    # ==============================================================================================

    if variant == (
        "Fixed Threshold"
    ):

        assert (
            edges
            ==
            ref[
                "fixed_threshold_edges"
            ]
        ), (
            f"{dataset}: Fixed Threshold edge control mismatch"
        )


        assert (
            reachable
            ==
            ref[
                "fixed_threshold_reachable"
            ]
        ), (
            f"{dataset}: Fixed Threshold reachability control mismatch"
        )


        print(
            "Frozen Cell-17 Fixed Threshold control: EXACT"
        )


    # ------------------------------------------------------------------------------------------
    # Save question-level result + number of retrieved paths for that question.
    # ------------------------------------------------------------------------------------------

    qcounts = pd.Series(
        CURRENT_PATH_COUNTS,
        dtype="int64"
    )


    perq_df = (
        perq_df.copy()
    )


    perq_df[
        "retrieved_path_count"
    ] = (
        perq_df[
            "question_id"
        ]
        .astype(
            str
        )
        .map(
            qcounts
        )
        .fillna(
            0
        )
        .astype(
            int
        )
    )


    perq_df[
        "ablation_variant"
    ] = variant


    perq_df.to_csv(
        perq_file,
        index=False
    )


    total_paths = int(
        sum(
            CURRENT_PATH_COUNTS.values()
        )
    )


    summary = {

        "dataset":
            dataset,

        "variant":
            variant,

        "edges_examined":
            edges,

        "ssr":
            ssr,

        "reachable_questions":
            reachable,

        "rog_reachable_questions":
            int(
                ref[
                    "rog_reachable"
                ]
            ),

        "answer_retention":
            ar,

        "active_hop_rows":
            int(
                row[
                    "active_hop_rows"
                ]
            ),

        "active_prefixes":
            int(
                row[
                    "active_prefixes"
                ]
            ),

        "candidate_branches":
            int(
                row[
                    "candidate_branches"
                ]
            ),

        "reachable_plans":
            int(
                row[
                    "reachable_plans"
                ]
            ),

        "decision_hops":
            int(
                row[
                    "decision_hops"
                ]
            ),

        "avg_requested_budget":
            float(
                row[
                    "avg_requested_budget"
                ]
            ),

        "avg_uncertainty":
            float(
                row[
                    "avg_uncertainty"
                ]
            ),

        "fully_tied_decisions":
            int(
                row[
                    "fully_tied_decisions"
                ]
            ),

        "peak_frontier":
            int(
                row[
                    "peak_frontier"
                ]
            ),

        "retrieved_paths":
            total_paths,

        "elapsed_min":
            elapsed
            /
            60.0,

        "retrieved_paths_file":
            str(
                path_file
            ),

        "per_question_file":
            str(
                perq_file
            ),
    }


    ALL_SUMMARY_ROWS.append(
        summary
    )


    ALL_PATH_FILES.append(
        path_file
    )


    ALL_PERQ_FILES.append(
        perq_file
    )


    print(
        f"edges = {edges:,}"
    )


    print(
        f"SSR   = {ssr:.6f}"
    )


    print(
        f"reach = "
        f"{reachable}/"
        f"{ref['rog_reachable']}"
    )


    print(
        f"AR    = {ar:.6f}"
    )


    print(
        f"paths = {total_paths:,}"
    )


    print(
        f"time  = {elapsed/60.0:.2f} min"
    )


    return summary


# ==================================================================================================
# 13. INSTALL THE EXACT DYNAMIC ROUTES
#     RUN 16 TEST EVALUATIONS
#     ALWAYS RESTORE THE FROZEN RUNTIME
# ==================================================================================================

SAVED_SCORE_BINDING = (
    GET_LOGITS_GLOBALS[
        "AFP_RUNTIME_SCORE_GROUP"
    ]
)


SAVED_SELECTOR_BINDING = (
    TRAVERSE_GLOBALS[
        "controlled_selection"
    ]
)


SAVED_TRAVERSE_BINDING = (
    RUN_GLOBALS[
        "traverse_controlled_method"
    ]
)


assert (
    SAVED_SCORE_BINDING
    is
    FROZEN_SCORE_GROUP
)


assert (
    SAVED_SELECTOR_BINDING
    is
    FROZEN_SELECTION
)


assert (
    SAVED_TRAVERSE_BINDING
    is
    FROZEN_TRAVERSE
)


try:

    # ==============================================================================================
    # CRITICAL FIX:
    #
    # The exact function used by get_group_logits is AFP_RUNTIME_SCORE_GROUP.
    #
    # DO NOT patch lowercase afp_runtime_score_group.
    # ==============================================================================================

    GET_LOGITS_GLOBALS[
        "AFP_RUNTIME_SCORE_GROUP"
    ] = final_ablation_score_router


    TRAVERSE_GLOBALS[
        "controlled_selection"
    ] = final_ablation_selector_router


    RUN_GLOBALS[
        "traverse_controlled_method"
    ] = capture_traverse


    assert (
        GET_LOGITS_GLOBALS[
            "AFP_RUNTIME_SCORE_GROUP"
        ]
        is
        final_ablation_score_router
    )


    assert (
        TRAVERSE_GLOBALS[
            "controlled_selection"
        ]
        is
        final_ablation_selector_router
    )


    assert (
        RUN_GLOBALS[
            "traverse_controlled_method"
        ]
        is
        capture_traverse
    )


    print(
        "\nCorrect uppercase scorer routing installed: PASSED"
    )


    print(
        "Correct selector routing installed: PASSED"
    )


    print(
        "Retrieved-path capture installed: PASSED"
    )


    # ==============================================================================================
    # 2 datasets × 8 variants = 16 final TEST evaluations.
    # ==============================================================================================

    for (
        dataset,
        plan_rows,
        q_rows,
        rog_ref,
        afp_spec,
        threshold_spec
    ) in [

        (
            "webqsp",
            WEB_PLAN_ROWS,
            WEB_Q_ROWS,
            WEB_ROG_REF,
            WEB_AFP_SPEC,
            WEB_THRESHOLD_SPEC
        ),

        (
            "cwq",
            CWQ_PLAN_ROWS,
            CWQ_Q_ROWS,
            CWQ_ROG_REF,
            CWQ_AFP_SPEC,
            CWQ_THRESHOLD_SPEC
        ),
    ]:

        for variant in (
            VARIANTS
        ):

            run_one(
                dataset=
                    dataset,

                variant=
                    variant,

                plan_rows=
                    plan_rows,

                q_rows=
                    q_rows,

                rog_ref=
                    rog_ref,

                afp_spec=
                    afp_spec,

                threshold_spec=
                    threshold_spec
            )


finally:

    # ==============================================================================================
    # ALWAYS restore exact Cell-17 bindings, even if a TEST variant fails.
    # ==============================================================================================

    GET_LOGITS_GLOBALS[
        "AFP_RUNTIME_SCORE_GROUP"
    ] = SAVED_SCORE_BINDING


    TRAVERSE_GLOBALS[
        "controlled_selection"
    ] = SAVED_SELECTOR_BINDING


    RUN_GLOBALS[
        "traverse_controlled_method"
    ] = SAVED_TRAVERSE_BINDING


    CURRENT_PATH_HANDLE = None


    print(
        "\nFrozen Cell-17 runtime bindings restored."
    )


assert (
    GET_LOGITS_GLOBALS[
        "AFP_RUNTIME_SCORE_GROUP"
    ]
    is
    FROZEN_SCORE_GROUP
)


assert (
    TRAVERSE_GLOBALS[
        "controlled_selection"
    ]
    is
    FROZEN_SELECTION
)


assert (
    RUN_GLOBALS[
        "traverse_controlled_method"
    ]
    is
    FROZEN_TRAVERSE
)


# ==================================================================================================
# 14. HARD POST-RUN ROUTING + DIMENSION AUDIT
# ==================================================================================================

print(
    "\n"
    +
    "=" * 150
)


print(
    "POST-RUN FEATURE-ABLATION ROUTING / DIMENSION AUDIT"
)


print(
    "=" * 150
)


for dataset in [
    "webqsp",
    "cwq",
]:

    for variant, cfg in (
        FEATURE_VARIANTS.items()
    ):

        key = (
            dataset,
            variant
        )


        calls = int(
            SCORE_CALLS[
                key
            ]
        )


        dims = sorted(
            OBSERVED_DIMS[
                key
            ]
        )


        logit_diff = bool(
            LOGIT_DIFF_SEEN[
                key
            ]
        )


        rank_diff = bool(
            RANK_DIFF_SEEN[
                key
            ]
        )


        print(

            f"{dataset:7s} | "

            f"{variant:18s} | "

            f"calls={calls:6d} | "

            f"observed_dims={dims} | "

            f"expected={cfg['dim']:2d} | "

            f"logits_differ_from_full={logit_diff} | "

            f"ranking_diff_seen={rank_diff}"
        )


        # ==========================================================================================
        # Gate 1:
        # the reduced-dimensional scorer MUST actually have been reached.
        # ==========================================================================================

        assert (
            calls
            >
            0
        ), (
            f"STOP: {dataset} {variant} "
            "reduced scorer was never called"
        )


        # ==========================================================================================
        # Gate 2:
        # every scoring invocation MUST use the intended reduced dimension.
        # ==========================================================================================

        assert (
            dims
            ==
            [
                cfg[
                    "dim"
                ]
            ]
        ), (
            f"STOP: {dataset} {variant} "
            f"used wrong dimension {dims}"
        )


        # ==========================================================================================
        # Gate 3:
        # prove this scorer is not silently producing the exact Full-AFP logits.
        #
        # This directly protects against the earlier failure where all feature
        # ablations effectively traveled through the Full AFP scorer.
        # ==========================================================================================

        assert (
            logit_diff
        ), (
            f"STOP: {dataset} {variant} never produced "
            "logits different from Full AFP. "
            "This is the exact failure mode we are preventing."
        )


# ------------------------------------------------------------------------------------------
# The custom Cell-15B no-uncertainty selector must actually execute.
# ------------------------------------------------------------------------------------------

for dataset in [
    "webqsp",
    "cwq",
]:

    calls = int(
        SELECTOR_CALLS[
            (
                dataset,
                "w/o Uncertainty"
            )
        ]
    )


    print(
        f"{dataset:7s} | "
        f"w/o Uncertainty selector calls="
        f"{calls}"
    )


    assert (
        calls
        >
        0
    ), (
        f"STOP: {dataset} w/o Uncertainty "
        "custom selector was never called"
    )


print(
    "All routing / reduced-dimension gates: PASSED"
)


# ==================================================================================================
# 15. FINAL 16-ROW SUMMARY + DELTAS VS FULL AFP
# ==================================================================================================

FINAL_TEST_ABLATION_SUMMARY_DF = (
    pd.DataFrame(
        ALL_SUMMARY_ROWS
    )
)


assert (
    len(
        FINAL_TEST_ABLATION_SUMMARY_DF
    )
    ==
    16
)


_dataset_order = {
    "webqsp": 0,
    "cwq": 1,
}


_variant_order = {

    v:
        i

    for i, v
    in enumerate(
        VARIANTS
    )
}


FINAL_TEST_ABLATION_SUMMARY_DF[
    "_d"
] = (
    FINAL_TEST_ABLATION_SUMMARY_DF[
        "dataset"
    ].map(
        _dataset_order
    )
)


FINAL_TEST_ABLATION_SUMMARY_DF[
    "_v"
] = (
    FINAL_TEST_ABLATION_SUMMARY_DF[
        "variant"
    ].map(
        _variant_order
    )
)


FINAL_TEST_ABLATION_SUMMARY_DF = (

    FINAL_TEST_ABLATION_SUMMARY_DF

    .sort_values(
        [
            "_d",
            "_v"
        ]
    )

    .drop(
        columns=[
            "_d",
            "_v"
        ]
    )

    .reset_index(
        drop=True
    )
)


FINAL_TEST_ABLATION_SUMMARY_DF[
    "delta_AR_vs_full"
] = np.nan


FINAL_TEST_ABLATION_SUMMARY_DF[
    "delta_SSR_vs_full"
] = np.nan


FINAL_TEST_ABLATION_SUMMARY_DF[
    "delta_edges_vs_full"
] = np.nan


for dataset in [
    "webqsp",
    "cwq",
]:

    mask = (
        FINAL_TEST_ABLATION_SUMMARY_DF[
            "dataset"
        ]
        ==
        dataset
    )


    full = (
        FINAL_TEST_ABLATION_SUMMARY_DF[

            mask

            &

            (
                FINAL_TEST_ABLATION_SUMMARY_DF[
                    "variant"
                ]
                ==
                "Full AFP"
            )
        ]
        .iloc[
            0
        ]
    )


    FINAL_TEST_ABLATION_SUMMARY_DF.loc[
        mask,
        "delta_AR_vs_full"
    ] = (

        FINAL_TEST_ABLATION_SUMMARY_DF.loc[
            mask,
            "answer_retention"
        ]

        -

        float(
            full[
                "answer_retention"
            ]
        )
    )


    FINAL_TEST_ABLATION_SUMMARY_DF.loc[
        mask,
        "delta_SSR_vs_full"
    ] = (

        FINAL_TEST_ABLATION_SUMMARY_DF.loc[
            mask,
            "ssr"
        ]

        -

        float(
            full[
                "ssr"
            ]
        )
    )


    FINAL_TEST_ABLATION_SUMMARY_DF.loc[
        mask,
        "delta_edges_vs_full"
    ] = (

        FINAL_TEST_ABLATION_SUMMARY_DF.loc[
            mask,
            "edges_examined"
        ]

        -

        int(
            full[
                "edges_examined"
            ]
        )
    )


# ==================================================================================================
# 16. SAVE PAPER / RQ3 ARTIFACTS + AUDIT MANIFEST
# ==================================================================================================

SUMMARY_PATH = (
    ABL_ROOT
    /
    "final_test_ablation_summary_FINAL_VERIFIED.csv"
)


RQ3_INPUT_PATH = (
    ABL_ROOT
    /
    "final_test_ablation_rq3_inputs_FINAL_VERIFIED.csv"
)


DIM_AUDIT_PATH = (
    ABL_ROOT
    /
    "feature_dimension_routing_audit_FINAL_VERIFIED.json"
)


MANIFEST_PATH = (
    ABL_ROOT
    /
    "final_test_ablation_manifest_FINAL_VERIFIED.json"
)


FINAL_TEST_ABLATION_SUMMARY_DF.to_csv(
    SUMMARY_PATH,
    index=False
)


FINAL_TEST_ABLATION_SUMMARY_DF[
    [
        "dataset",
        "variant",
        "answer_retention",
        "ssr",
        "edges_examined",
        "reachable_questions",
        "rog_reachable_questions",
        "retrieved_paths",
        "delta_AR_vs_full",
        "delta_SSR_vs_full",
        "delta_edges_vs_full",
        "retrieved_paths_file",
        "per_question_file",
    ]
].to_csv(
    RQ3_INPUT_PATH,
    index=False
)


# ------------------------------------------------------------------------------------------
# Explicit software-scientific audit artifact.
# ------------------------------------------------------------------------------------------

DIM_AUDIT = {

    "full_feature_dim":
        27,

    "feature_variants": {

        variant: {

            "internal":
                cfg[
                    "internal"
                ],

            "removed_indices":
                cfg[
                    "removed"
                ],

            "kept_indices":
                cfg[
                    "keep"
                ],

            "expected_input_dim":
                cfg[
                    "dim"
                ],

            "checkpoint_sha256": {

                dataset:
                    FEATURE_CHECKPOINT_SHA[
                        dataset
                    ][
                        variant
                    ]

                for dataset
                in [
                    "webqsp",
                    "cwq"
                ]
            },
        }

        for variant, cfg
        in FEATURE_VARIANTS.items()
    },

    "observed": {

        f"{dataset}|{variant}": {

            "score_calls":
                int(
                    SCORE_CALLS[
                        (
                            dataset,
                            variant
                        )
                    ]
                ),

            "observed_dims":
                sorted(
                    OBSERVED_DIMS[
                        (
                            dataset,
                            variant
                        )
                    ]
                ),

            "logits_differ_from_full":
                bool(
                    LOGIT_DIFF_SEEN[
                        (
                            dataset,
                            variant
                        )
                    ]
                ),

            "ranking_difference_seen":
                bool(
                    RANK_DIFF_SEEN[
                        (
                            dataset,
                            variant
                        )
                    ]
                ),
        }

        for dataset
        in [
            "webqsp",
            "cwq"
        ]

        for variant
        in FEATURE_VARIANTS
    },

    "uppercase_runtime_binding":
        "AFP_RUNTIME_SCORE_GROUP",

    "test_training":
        False,

    "test_tuning":
        False,

    "test_model_selection":
        False,
}


DIM_AUDIT_PATH.write_text(

    json.dumps(
        DIM_AUDIT,
        indent=2,
        ensure_ascii=False
    ),

    encoding="utf-8"
)


artifact_hashes = {

    "summary":
        sha256_file(
            SUMMARY_PATH
        ),

    "rq3_input":
        sha256_file(
            RQ3_INPUT_PATH
        ),

    "dimension_audit":
        sha256_file(
            DIM_AUDIT_PATH
        ),

    "retrieved_paths": {

        str(
            p
        ):
            sha256_file(
                p
            )

        for p in (
            ALL_PATH_FILES
        )
    },

    "per_question": {

        str(
            p
        ):
            sha256_file(
                p
            )

        for p in (
            ALL_PERQ_FILES
        )
    },
}


MANIFEST = {

    "experiment":
        "Final frozen TEST AFP ablation study",

    "version":
        "FINAL_VERIFIED_one_cell_uppercase_routing_dimension_safe_v1",

    # ----------------------------------------------------------------------------------------------
    # Immutable development freeze.
    # ----------------------------------------------------------------------------------------------

    "scientific_freeze_sha256":
        EXPECTED_FREEZE_SHA,

    # ----------------------------------------------------------------------------------------------
    # Current Cell-17I provenance.
    # Its execution-manifest hash may legitimately differ between reproducibility runs.
    # ----------------------------------------------------------------------------------------------

    "source_cell17_manifest":
        str(
            globals().get(
                "CELL17_MANIFEST_PATH",
                ""
            )
        ),

    "source_cell17_manifest_sha256":
        str(
            globals().get(
                "CELL17_MANIFEST_SHA256",
                ""
            )
        ),

    "variants":
        VARIANTS,

    # ----------------------------------------------------------------------------------------------
    # Exact feature-removal contracts.
    # ----------------------------------------------------------------------------------------------

    "feature_dimensions": {

        variant:
            cfg[
                "dim"
            ]

        for variant, cfg
        in FEATURE_VARIANTS.items()
    },

    "feature_keep_indices": {

        variant:
            cfg[
                "keep"
            ]

        for variant, cfg
        in FEATURE_VARIANTS.items()
    },

    "feature_removed_indices": {

        variant:
            cfg[
                "removed"
            ]

        for variant, cfg
        in FEATURE_VARIANTS.items()
    },

    "feature_checkpoint_sha256":
        FEATURE_CHECKPOINT_SHA,

    # ----------------------------------------------------------------------------------------------
    # Exact selector-component causal definitions.
    # ----------------------------------------------------------------------------------------------

    "selector_ablation_contracts": {

        "w/o Uncertainty":
            (
                "Cell-15B exact: T unchanged; "
                "normalized entropy still measured diagnostically; "
                "gamma=gamma_min on non-tied decisions; "
                "fully tied decisions retain all with u=1,gamma=1"
            ),

        "w/o Temperature":
            (
                "Cell-15B exact: T=1.0; "
                "normalized-entropy uncertainty adaptation retained"
            ),
    },

    # ----------------------------------------------------------------------------------------------
    # Software-routing integrity.
    # ----------------------------------------------------------------------------------------------

    "routing": {

        "get_group_logits_binding_patched":
            "AFP_RUNTIME_SCORE_GROUP",

        "feature_scorer_dict_contract_preserved":
            True,

        "separate_run_controlled_dataset_call_per_dataset_variant":
            True,

        "score_cache_cross_variant_leakage_possible":
            False,
    },

    # ----------------------------------------------------------------------------------------------
    # Frozen traversal controls.
    # ----------------------------------------------------------------------------------------------

    "controls": {

        "full_afp_cell17_exact":
            True,

        "fixed_threshold_cell17_exact":
            True,

        "final_hop_protection":
            True,

        "singleton_bypass":
            True,
    },

    # ----------------------------------------------------------------------------------------------
    # Leakage / TEST-use declaration.
    # ----------------------------------------------------------------------------------------------

    "leakage": {

        "test_training":
            False,

        "test_tuning":
            False,

        "test_model_selection":
            False,

        "gold_used_by_scorer":
            False,

        "gold_used_by_selector":
            False,

        "gold_used_posthoc_for_reachability":
            True,
    },

    "routing_dimension_audit":
        DIM_AUDIT,

    "artifact_hashes":
        artifact_hashes,
}


MANIFEST_PATH.write_text(

    json.dumps(
        MANIFEST,
        indent=2,
        ensure_ascii=False
    ),

    encoding="utf-8"
)


MANIFEST_SHA256 = (
    sha256_file(
        MANIFEST_PATH
    )
)


# ------------------------------------------------------------------------------------------
# Physical artifact-integrity gates.
# ------------------------------------------------------------------------------------------

for p in (

    [
        SUMMARY_PATH,
        RQ3_INPUT_PATH,
        DIM_AUDIT_PATH,
        MANIFEST_PATH,
    ]

    +

    ALL_PATH_FILES

    +

    ALL_PERQ_FILES
):

    assert (
        p.exists()
    )


    assert (
        p.stat().st_size
        >
        0
    )


# ==================================================================================================
# 17. FINAL FLAGS + PAPER-READY DISPLAY
# ==================================================================================================

FINAL_TEST_ABLATION_COMPLETE = (
    True
)


RQ2_ABLATION_TEST_COMPLETE = (
    True
)


RQ3_RETRIEVED_PATHS_READY = (
    True
)


print(
    "\n"
    +
    "=" * 150
)


print(
    "FINAL VERIFIED TEST ABLATION RESULTS"
)


print(
    "=" * 150
)


DISPLAY_COLS = [

    "dataset",

    "variant",

    "answer_retention",

    "ssr",

    "edges_examined",

    "reachable_questions",

    "retrieved_paths",

    "delta_AR_vs_full",

    "delta_SSR_vs_full",

    "delta_edges_vs_full",
]


print(

    FINAL_TEST_ABLATION_SUMMARY_DF[
        DISPLAY_COLS
    ].to_string(

        index=False,

        formatters={

            "answer_retention":
                lambda x:
                    f"{x:.6f}",

            "ssr":
                lambda x:
                    f"{x:.6f}",

            "delta_AR_vs_full":
                lambda x:
                    f"{x:+.6f}",

            "delta_SSR_vs_full":
                lambda x:
                    f"{x:+.6f}",
        }
    )
)


print(
    "\n"
    +
    "=" * 150
)


print(
    "FINAL ABLATION STUDY COMPLETE — READY FOR RQ3"
)


print(
    "=" * 150
)


print(
    "FINAL_TEST_ABLATION_COMPLETE:",
    FINAL_TEST_ABLATION_COMPLETE
)


print(
    "RQ2_ABLATION_TEST_COMPLETE: ",
    RQ2_ABLATION_TEST_COMPLETE
)


print(
    "RQ3_RETRIEVED_PATHS_READY:  ",
    RQ3_RETRIEVED_PATHS_READY
)


print(
    "TEST training:               NO"
)


print(
    "TEST tuning:                 NO"
)


print(
    "TEST model selection:        NO"
)


print(
    "Full AFP control:            EXACT"
)


print(
    "Fixed Threshold control:     EXACT"
)


print(
    "Feature dims verified:       19 / 20 / 19 / 23"
)


print(
    "Scorer route verified:       AFP_RUNTIME_SCORE_GROUP (UPPERCASE)"
)


print(
    "\nSummary:",
    SUMMARY_PATH
)


print(
    "RQ3 input table:",
    RQ3_INPUT_PATH
)


print(
    "Retrieved paths:",
    PATH_ROOT
)


print(
    "Dimension audit:",
    DIM_AUDIT_PATH
)


print(
    "Manifest:",
    MANIFEST_PATH
)


print(
    "Manifest SHA256:",
    MANIFEST_SHA256
)
